# Exploração — Dataset Expandido: top 5k + 6k genes ausentes no Mathys

**Objetivo:** Testar se incluir os ~6279 genes presentes no Fujita mas ausentes no Mathys muda as métricas de recuperação da Rede de Hopfield. Três experimentos são comparados:

| Exp | Dataset | Estratégia de protótipos | Padrões/classe |
|-----|---------|--------------------------|----------------|
| **A** baseline | top 5000 genes | KMeans nc=10, k=1 | 10 |
| **B** | top 5k + 6k genes | 70 células mais próximas do centroide da classe (sem KMeans) | 70 |
| **C** | top 5k + 6k genes | KMeans nc=10, k=20 (média das 20 mais próximas) | 10 |

---
> ⚠️ As matrizes expandidas (~11k genes × ~40k células) exigem ~2–4 GB de RAM. Execute em ambiente com memória suficiente.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import polars as pl

sys.path.insert(0, '.')

from src.treinamento.hopfield_utils import closervects
from src.treinamento.projetor_sweep import ProjetorSWeP
from src.treinamento.extrator_padroes import ExtratorPadroesSubcluster
from src.treinamento.hopfield import ModernHopfieldNetwork
from src.treinamento.avaliador_hopfield import AvaliadorHopfield
from src.treinamento.carregador_dados_fujita import CarregadorDadosFujita

CLASSES = [1, 3, 4, 5, 6, 7]

PATH_F_ALINHADO  = 'outputs/alinhamento/adataF_binarizado_alinhado/adataF_binarizado_alinhado.txt'
PATH_M_ALINHADO  = 'outputs/alinhamento/adataM_binarizado_alinhado/adataM_binarizado_alinhado.txt'
PATH_TOP5K       = 'outputs/top_genes/top5000_frequentes.csv'
PATH_TRACKING    = 'outputs/alinhamento/tracking_genes_adicionados_mathys.csv'
PATH_LABELS_F    = 'imputs/cell_types_binarioF.txt'
PATH_LABELS_M    = 'imputs/celltypeBinMparcial.csv'
PATH_F_BASE      = 'outputs/treinamento/adataF_binarizado_alinhado_top5000.txt'
PATH_M_BASE      = 'outputs/treinamento/adataM_binarizado_alinhado_top5000.txt'
PATH_SWEEP_BASE  = 'outputs/treinamento/matriz_reduzida_sweepF.csv'
PATH_F_EXP       = 'outputs/treinamento/adataF_alinhado_top5k_mais6k.txt'
PATH_M_EXP       = 'outputs/treinamento/adataM_alinhado_top5k_mais6k.txt'

os.makedirs('outputs/hopfield', exist_ok=True)

## Seção 1 — Construção do gene set expandido

União dos top 5000 genes mais frequentes no Fujita com os ~6279 genes ausentes no Mathys.

In [2]:
top5k = pl.read_csv(PATH_TOP5K)['gene'].to_list()
ausentes = pl.read_csv(PATH_TRACKING)['ensembl_id'].to_list()

# Union preservando ordem: top5k primeiro, depois genes ausentes não duplicados
genes_expandidos = list(dict.fromkeys(top5k + ausentes))

print(f'top 5000 genes      : {len(top5k)}')
print(f'genes ausentes (M)  : {len(ausentes)}')
print(f'genes expandidos    : {len(genes_expandidos)}')
print(f'overlap (esperado 0): {len(set(top5k) & set(ausentes))}')

top 5000 genes      : 5000
genes ausentes (M)  : 6279
genes expandidos    : 11185
overlap (esperado 0): 94


## Seção 2 — Filtragem das matrizes alinhadas para o gene set expandido

Usa `polars.scan_csv + sink_csv` (streaming) para suportar os arquivos grandes sem carregar tudo na memória. Skip-if-exists incluído.

In [3]:
def filtrar_matriz_expandida(path_source, path_dest, genes):
    if os.path.exists(path_dest):
        print(f'[skip] já existe: {path_dest}')
        return
    # Verifica quais colunas estão de fato no arquivo fonte
    with open(path_source, encoding='utf-8') as f:
        colunas_disponiveis = set(f.readline().strip().split(','))
    genes_validos = [g for g in genes if g in colunas_disponiveis]
    faltando = len(genes) - len(genes_validos)
    if faltando:
        print(f'  ⚠️  {faltando} genes do gene set não encontrados em {os.path.basename(path_source)}')
    print(f'Filtrando {path_source}  →  {len(genes_validos)} colunas...')
    path_tmp = path_dest + '.tmp'
    (
        pl.scan_csv(path_source)
        .select(genes_validos)
        .sink_csv(path_tmp)
    )
    os.rename(path_tmp, path_dest)
    print(f'Salvo: {path_dest}')

filtrar_matriz_expandida(PATH_F_ALINHADO, PATH_F_EXP, genes_expandidos)
filtrar_matriz_expandida(PATH_M_ALINHADO, PATH_M_EXP, genes_expandidos)

[skip] já existe: outputs/treinamento/adataF_alinhado_top5k_mais6k.txt
[skip] já existe: outputs/treinamento/adataM_alinhado_top5k_mais6k.txt


In [ ]:
# Verificação de shapes — detecta problemas de dimensão antes de carregar tudo na RAM
def verificar_arquivo(path, n_celulas_esperado, n_genes_esperado):
    with open(path, encoding='utf-8') as f:
        cols = len(f.readline().strip().split(','))
        rows = sum(1 for _ in f)
    ok_c = '✓' if rows == n_celulas_esperado else f'⚠️ esperado {n_celulas_esperado}, obtido {rows}'
    ok_g = '✓' if cols == n_genes_esperado    else f'⚠️ esperado {n_genes_esperado}, obtido {cols}'
    print(f'{os.path.basename(path)}: {rows} células {ok_c} | {cols} genes {ok_g}')

if os.path.exists(PATH_F_EXP):
    verificar_arquivo(PATH_F_EXP, 40913, len(genes_expandidos))
if os.path.exists(PATH_M_EXP):
    verificar_arquivo(PATH_M_EXP, 45663, len(genes_expandidos))

## Seção 3 — Carregamento de dados e projeção SWeeP

### 3a — Rótulos (comuns a todos os experimentos)

In [ ]:
labels_F = np.loadtxt(PATH_LABELS_F, dtype=int).ravel()
labels_M = np.loadtxt(PATH_LABELS_M, dtype=int).ravel()

print(f'Rótulos Fujita : {labels_F.shape}  | tipos: {np.unique(labels_F)}')
print(f'Rótulos Mathys : {labels_M.shape}  | tipos: {np.unique(labels_M)}')

### 3b — Baseline: top 5000 genes (Exp. A)

In [ ]:
dados_base = CarregadorDadosFujita(
    path_matriz=PATH_F_BASE,
    path_genes=PATH_TOP5K,
    path_labels=PATH_LABELS_F,
    path_sweep=PATH_SWEEP_BASE,
).carregar()

# PCA sem centralização sobre o SWeeP pré-computado
proj_base = ProjetorSWeP(n_features=dados_base.W0.shape[1], n_componentes=600)
proj_base.usar_sweep_precomputado(dados_base.Wswp).aplicar_pca()
Wswp_F_base = proj_base.Wpc  # (40913, 600)

print('Carregando matriz Mathys baseline...')
W0_M_base = pl.read_csv(PATH_M_BASE).to_numpy().astype(np.float32)
print(f'W0_M_base shape: {W0_M_base.shape}')

### 3c — Dataset expandido: top 5k + 6k genes (Exp. B e C)

In [ ]:
print('Carregando matriz Fujita expandida...')
W0_F_exp = pl.read_csv(PATH_F_EXP).to_numpy().astype(np.float32)
print(f'W0_F_exp shape: {W0_F_exp.shape}')

print('Carregando matriz Mathys expandida...')
W0_M_exp = pl.read_csv(PATH_M_EXP).to_numpy().astype(np.float32)
print(f'W0_M_exp shape: {W0_M_exp.shape}')

In [ ]:
# Verificação: genes ausentes no Mathys devem ter valor 0.5 nas colunas adicionadas
idx_ausente_exemplo = genes_expandidos.index(ausentes[0])
vals_ausente = np.unique(W0_M_exp[:, idx_ausente_exemplo])
print(f'Valores únicos na coluna de um gene ausente ({ausentes[0]}): {vals_ausente}')
# Esperado: [0.5]

In [ ]:
# Projeção SWeeP + PCA no espaço expandido (base QR sintética, reprodutível via seed)
proj_exp = ProjetorSWeP(n_features=W0_F_exp.shape[1], n_componentes=600, seed=42)
proj_exp.gerar_base().projetar(W0_F_exp).aplicar_pca()
Wswp_F_exp = proj_exp.Wpc  # (40913, 600)
print(f'SWeeP expandido Fujita: {Wswp_F_exp.shape}')

## Seção 4 — Experimentos

### Experimento A — Baseline (top 5000 genes, KMeans nc=10, k=1)

In [5]:
extrator_A = ExtratorPadroesSubcluster(
    dados_base.W0, labels_F, classes=CLASSES, nc=10, k=1, seed=42
)
extrator_A.extrair(Wswp_F_base)
print(extrator_A)

NameError: name 'dados_base' is not defined

In [ ]:
rede_A = ModernHopfieldNetwork(beta=8.0, n_iters=1, binary=True, threshold=0.5)
rede_A.store(extrator_A.padroes)

W_rec_A = rede_A.retrieve(W0_M_base, batch_size=4096)
rede_A.salvar('outputs/hopfield/rede_expA_baseline.pt')

In [ ]:
classes_A = sorted({m[0] for m in extrator_A.meta})
av_A = AvaliadorHopfield(
    padroes=extrator_A.padroes,
    classes=classes_A,
    nc=10,
    meta=extrator_A.meta,
)
av_A.avaliar(W_rec_A, labels_M)
av_A.plotar(titulo='Exp A — Baseline top5k, nc=10, k=1')

### Experimento B — 70 células mais representativas por classe (sem KMeans, dataset expandido)

Para cada classe, calcula o centroide como a média de todas as células da classe no espaço SWeeP,
depois seleciona as 70 células reais mais próximas desse centroide. Cada célula vira um padrão individual.

In [6]:
N_REP = 70
padroes_B = []
meta_B = []

for cls in CLASSES:
    ids_cls = np.where(labels_F == cls)[0]
    Wswp_cls = Wswp_F_exp[ids_cls]
    W0_cls   = W0_F_exp[ids_cls]

    centroide = Wswp_cls.mean(axis=0)  # centroide da classe inteira
    idxs = closervects(Wswp_cls, centroide, k=N_REP)  # 70 mais próximos (array)

    for i in idxs:
        padroes_B.append(W0_cls[i])
        meta_B.append((cls, int(ids_cls[i])))

    print(f'  classe {cls}: {len(ids_cls)} células → {N_REP} representantes selecionados')

padroes_B = np.vstack(padroes_B).astype(np.float32)
print(f'\npadroes_B shape: {padroes_B.shape}  ({len(CLASSES)} classes × {N_REP} rep = {len(CLASSES)*N_REP})')

NameError: name 'labels_F' is not defined

In [ ]:
rede_B = ModernHopfieldNetwork(beta=8.0, n_iters=1, binary=True, threshold=0.5)
rede_B.store(padroes_B)

W_rec_B = rede_B.retrieve(W0_M_exp, batch_size=4096)
rede_B.salvar('outputs/hopfield/rede_expB_70rep.pt')

In [ ]:
classes_B = sorted({m[0] for m in meta_B})
av_B = AvaliadorHopfield(
    padroes=padroes_B,
    classes=classes_B,
    nc=N_REP,
    meta=meta_B,
)
av_B.avaliar(W_rec_B, labels_M)
av_B.plotar(titulo='Exp B — top5k+6k, 70 rep/classe sem KMeans')

### Experimento C — KMeans nc=10, k=20 (protótipo = média das 20 mais próximas, dataset expandido)

In [ ]:
extrator_C = ExtratorPadroesSubcluster(
    W0_F_exp, labels_F, classes=CLASSES, nc=10, k=20, seed=42
)
extrator_C.extrair(Wswp_F_exp)
print(extrator_C)

In [ ]:
rede_C = ModernHopfieldNetwork(beta=8.0, n_iters=1, binary=True, threshold=0.5)
rede_C.store(extrator_C.padroes)

W_rec_C = rede_C.retrieve(W0_M_exp, batch_size=4096)
rede_C.salvar('outputs/hopfield/rede_expC_k20.pt')

In [ ]:
classes_C = sorted({m[0] for m in extrator_C.meta})
av_C = AvaliadorHopfield(
    padroes=extrator_C.padroes,
    classes=classes_C,
    nc=10,
    meta=extrator_C.meta,
)
av_C.avaliar(W_rec_C, labels_M)
av_C.plotar(titulo='Exp C — top5k+6k, nc=10, k=20')

## Seção 5 — Tabela comparativa de métricas

In [ ]:
resultados = pd.DataFrame([
    {
        'experimento'      : 'A_baseline_5k_nc10_k1',
        'n_genes'          : dados_base.W0.shape[1],
        'n_padroes'        : len(extrator_A.padroes),
        'acuracia'         : av_A.acuracia,
        'f1_macro'         : av_A.f1_macro,
        'f1_weighted'      : av_A.f1_weighted,
        'taxa_reconstrucao': av_A.taxa_reconstrucao,
        'semelhanca_media' : av_A.semelhanca_media,
    },
    {
        'experimento'      : 'B_5k+6k_70rep_semKMeans',
        'n_genes'          : W0_F_exp.shape[1],
        'n_padroes'        : len(padroes_B),
        'acuracia'         : av_B.acuracia,
        'f1_macro'         : av_B.f1_macro,
        'f1_weighted'      : av_B.f1_weighted,
        'taxa_reconstrucao': av_B.taxa_reconstrucao,
        'semelhanca_media' : av_B.semelhanca_media,
    },
    {
        'experimento'      : 'C_5k+6k_nc10_k20',
        'n_genes'          : W0_F_exp.shape[1],
        'n_padroes'        : len(extrator_C.padroes),
        'acuracia'         : av_C.acuracia,
        'f1_macro'         : av_C.f1_macro,
        'f1_weighted'      : av_C.f1_weighted,
        'taxa_reconstrucao': av_C.taxa_reconstrucao,
        'semelhanca_media' : av_C.semelhanca_media,
    },
])

resultados['acuracia_%']          = (resultados['acuracia'] * 100).round(2)
resultados['taxa_reconstrucao_%'] = (resultados['taxa_reconstrucao'] * 100).round(2)

os.makedirs('outputs/hopfield', exist_ok=True)
resultados.to_csv('outputs/hopfield/comparacao_experimentos.csv', index=False)

display(resultados[['experimento', 'n_genes', 'n_padroes',
                     'acuracia_%', 'f1_macro', 'f1_weighted', 'taxa_reconstrucao_%', 'semelhanca_media']])

## Notas e próximos passos

- Se **Exp B > Exp A**: os 6k genes ausentes adicionam sinal discriminativo
- Se **Exp B ≈ Exp A**: o espaço gênico extra não ajuda (0.5 é neutro para a rede)
- Se **Exp C > Exp A**: suavizar os protótipos com k=20 reduz ruído e melhora generalização
- Próximo passo sugerido: variar `beta` (ex: 4, 8, 16) e `n_iters` (1, 2, 3) para os melhores experimentos

## Seção 6 — Grid Search: beta × n_iters

Varia `beta` ∈ {4, 8, 16} e `n_iters` ∈ {1, 2, 3} para os 3 experimentos.
**Reutiliza** os padrões e matrizes já na memória — sem re-executar dados nem SWeeP.

| Parâmetro | Papel | Grid |
|---|---|---|
| `beta` | Temperatura do softmax — maior = winner-takes-all mais forte | 4, 8, 16 |
| `n_iters` | Iterações de atualização da memória | 1, 2, 3 |

**Total:** 3 experimentos × 9 combinações = 27 runs (8 novos + 3 já calculados).

In [ ]:
betas       = [4.0, 8.0, 16.0]
n_iteracoes = [1, 2, 3]

experimentos_sweep = {
    'A_baseline_5k': {
        'padroes': extrator_A.padroes, 'meta': extrator_A.meta,
        'W_query': W0_M_base,          'classes': classes_A,
    },
    'B_5k+6k_70rep': {
        'padroes': padroes_B,          'meta': meta_B,
        'W_query': W0_M_exp,           'classes': classes_B,
    },
    'C_5k+6k_nc10_k20': {
        'padroes': extrator_C.padroes, 'meta': extrator_C.meta,
        'W_query': W0_M_exp,           'classes': classes_C,
    },
}

registros = []

for exp_nome, cfg in experimentos_sweep.items():
    for beta in betas:
        for n_iter in n_iteracoes:
            if beta == 8.0 and n_iter == 1:
                continue  # resultado já existe em av_A / av_B / av_C
            print(f'[{exp_nome}] beta={beta}, n_iters={n_iter}...')
            rede = ModernHopfieldNetwork(beta=beta, n_iters=n_iter, binary=True, threshold=0.5)
            rede.store(cfg['padroes'])
            W_rec = rede.retrieve(cfg['W_query'], batch_size=4096)
            av = AvaliadorHopfield(cfg['padroes'], cfg['classes'], meta=cfg['meta'])
            av.avaliar(W_rec, labels_M)
            registros.append({
                'experimento'      : exp_nome,
                'beta'             : beta,
                'n_iters'          : n_iter,
                'acuracia'         : av.acuracia,
                'f1_macro'         : av.f1_macro,
                'f1_weighted'      : av.f1_weighted,
                'taxa_reconstrucao': av.taxa_reconstrucao,
                'semelhanca_media' : av.semelhanca_media,
            })

# Incorporar resultados já calculados (beta=8, n_iters=1)
for av_obj, nome in [(av_A, 'A_baseline_5k'),
                     (av_B, 'B_5k+6k_70rep'),
                     (av_C, 'C_5k+6k_nc10_k20')]:
    registros.append({
        'experimento'      : nome,
        'beta'             : 8.0,
        'n_iters'          : 1,
        'acuracia'         : av_obj.acuracia,
        'f1_macro'         : av_obj.f1_macro,
        'f1_weighted'      : av_obj.f1_weighted,
        'taxa_reconstrucao': av_obj.taxa_reconstrucao,
        'semelhanca_media'  : av_obj.semelhanca_media,
    })

df_sweep = (pd.DataFrame(registros)
              .sort_values(['experimento', 'beta', 'n_iters'])
              .reset_index(drop=True))
df_sweep.to_csv('outputs/hopfield/sweep_hiperparametros.csv', index=False)
display(df_sweep)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

metricas_plot = ['acuracia', 'f1_macro', 'f1_weighted', 'taxa_reconstrucao']

os.makedirs('outputs/hopfield', exist_ok=True)

for exp_nome in experimentos_sweep:
    df_exp = df_sweep[df_sweep['experimento'] == exp_nome]
    fig, axes = plt.subplots(1, len(metricas_plot), figsize=(16, 3))
    fig.suptitle(exp_nome, fontsize=13)
    for ax, metrica in zip(axes, metricas_plot):
        pivot = df_exp.pivot(index='n_iters', columns='beta', values=metrica)
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='Blues', ax=ax,
                    linewidths=0.5, cbar=False)
        ax.set_title(metrica)
    plt.tight_layout()
    plt.savefig(f'outputs/hopfield/heatmap_{exp_nome}.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Melhor combinação por f1_macro para cada experimento
melhor = (df_sweep
          .sort_values('f1_macro', ascending=False)
          .groupby('experimento')
          .first()
          [['beta', 'n_iters', 'acuracia', 'f1_macro', 'f1_weighted', 'taxa_reconstrucao']])
print('=== Melhor beta × n_iters por experimento (f1_macro) ===')
display(melhor)

## Seção 7 — Análise de Reconstrução nos Genes Injetados

Mede a qualidade de reconstrução separadamente para dois grupos de genes:
- **top5k** (presentes no Mathys, valores reais {0,1})
- **injetados** (ausentes no Mathys, preenchidos com 0.5 antes da recuperação)

Só se aplica a Exp B e C (dataset expandido). Responde: a rede consegue inferir os valores corretos dos genes ausentes?

In [ ]:
n_top5k       = 5000
idx_top5k     = np.arange(n_top5k)                       # colunas 0-4999 (genes presentes)
idx_injetados = np.arange(n_top5k, W0_F_exp.shape[1])   # colunas 5000-~11278 (injetados com 0.5)

def analisar_reconstrucao_por_grupo(W_rec, padroes, meta, classes, labels):
    """Semelhança média e taxa de reconstrução exata para 3 grupos de genes:
    total, top-5k (presentes no Mathys), injetados (ausentes → 0.5).

    O protótipo mais próximo é sempre encontrado no espaço COMPLETO (igual ao AvaliadorHopfield),
    mas o Hamming é medido separadamente por grupo de colunas.
    """
    pattern_classes = np.array([m[0] for m in meta])
    W_f    = W_rec.astype(np.float64)
    perf_f = padroes.astype(np.float64)

    a2 = (W_f ** 2).sum(axis=1, keepdims=True)
    b2 = (perf_f ** 2).sum(axis=1, keepdims=True).T
    idx_proto  = (a2 + b2 - 2 * (W_f @ perf_f.T)).argmin(axis=1)
    prototipos = perf_f[idx_proto]

    mask = np.isin(labels, classes)
    pred = pattern_classes[idx_proto]

    resultados = {}
    for nome_grupo, idx_grupo in [('total',     np.arange(W_f.shape[1])),
                                   ('top5k',     idx_top5k),
                                   ('injetados', idx_injetados)]:
        W_g = W_f[mask][:, idx_grupo]
        P_g = prototipos[mask][:, idx_grupo]
        h = (W_g != P_g).mean(axis=1)
        resultados[nome_grupo] = {
            'semelhanca_media'  : float((1 - h).mean()),
            'taxa_rec_exata'    : float((h == 0).mean()),
            'hamming_medio'     : float(h.mean()),
        }
    return resultados, pred[mask], labels[mask]

print(f'idx_injetados: {len(idx_injetados)} genes (colunas {idx_injetados[0]} a {idx_injetados[-1]})')

In [ ]:
res_B, pred_B, ytrue_B = analisar_reconstrucao_por_grupo(
    W_rec_B, padroes_B, meta_B, classes_B, labels_M)
res_C, pred_C, ytrue_C = analisar_reconstrucao_por_grupo(
    W_rec_C, extrator_C.padroes, extrator_C.meta, classes_C, labels_M)

linhas = []
for exp_nome, res in [('B_5k+6k_70rep', res_B), ('C_5k+6k_nc10_k20', res_C)]:
    for grupo, vals in res.items():
        linhas.append({'experimento': exp_nome, 'grupo_genes': grupo, **vals})

df_grupos = pd.DataFrame(linhas)
df_grupos.to_csv('outputs/hopfield/reconstrucao_por_grupo_genes.csv', index=False)
display(df_grupos.pivot_table(index='grupo_genes', columns='experimento',
                               values=['semelhanca_media', 'taxa_rec_exata']).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (exp_nome, res) in zip(axes,
        [('Exp B — 70rep', res_B), ('Exp C — nc10 k20', res_C)]):
    grupos = ['top5k\n(presentes)', 'injetados\n(0.5 → {0,1})', 'total']
    chaves = ['top5k', 'injetados', 'total']
    vals   = [res[k]['semelhanca_media'] for k in chaves]
    cores  = ['#2196F3', '#FF7043', '#9E9E9E']
    bars = ax.bar(grupos, vals, color=cores, edgecolor='black', linewidth=0.5)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Semelhança média ao protótipo')
    ax.set_title(exp_nome)
    ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, label='limiar 0.5')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.015,
                f'{v:.3f}', ha='center', fontsize=10)

plt.suptitle('Qualidade de reconstrução: genes presentes vs injetados', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/hopfield/reconstrucao_por_grupo_genes.png', dpi=120, bbox_inches='tight')
plt.show()

# --- Semelhança por classe (somente genes injetados) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (exp_nome, W_rec, padroes, meta, classes) in zip(axes, [
        ('Exp B', W_rec_B, padroes_B, meta_B, classes_B),
        ('Exp C', W_rec_C, extrator_C.padroes, extrator_C.meta, classes_C)]):
    pattern_classes_ax = np.array([m[0] for m in meta])
    W_full   = W_rec.astype(np.float64)
    perf_full = padroes.astype(np.float64)
    idx_p = ((W_full ** 2).sum(1, keepdims=True) +
              (perf_full ** 2).sum(1, keepdims=True).T -
              2 * (W_full @ perf_full.T)).argmin(axis=1)
    W_inj      = W_full[:, idx_injetados]
    perf_inj   = perf_full[idx_p][:, idx_injetados]
    mask_ax    = np.isin(labels_M, classes)
    h_inj      = (W_inj[mask_ax] != perf_inj[mask_ax]).mean(axis=1)
    y_true_ax  = labels_M[mask_ax]
    sim_classe = {cls: float((1 - h_inj[y_true_ax == cls]).mean())
                  for cls in classes if (y_true_ax == cls).any()}
    ax.bar(list(sim_classe.keys()), list(sim_classe.values()), color='#FF7043')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Tipo celular')
    ax.set_ylabel('Semelhança média (injetados)')
    ax.set_title(f'{exp_nome} — por classe')
    ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('outputs/hopfield/reconstrucao_injetados_por_classe.png', dpi=120, bbox_inches='tight')
plt.show()